# Components lab 06: small component lab

This lab wires a tiny link: an entangled-pair source sends one member toward memory and the other toward a detector. The point is to watch ports, channels, qstate refs, memory reports, and detector reports line up in one run.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field

from simyuj.components import ACTION_TRANSMIT_QUANTUM, MEMORY_ABSORB, MEMORY_MEASURE, Port, PortDelivery, PortDirection, PortKind, QuantumChannel, QuantumMemory, connect_ports
from simyuj.components.detectors.detector_array import DetectorArray
from simyuj.components.detectors.primitives.actions import ACTION_DETECT_SIGNAL
from simyuj.components.detectors.primitives.params import SinglePhotonDetectorParams
from simyuj.components.detectors.primitives.reports import DetectionReport
from simyuj.components.detectors.single_photon import SinglePhotonDetector
from simyuj.components.memories import MemoryMeasureRequest
from simyuj.components.sources import EntangledPairSource
from simyuj.engine import Component, Event, Timeline
from simyuj.qstate.noise import depolarizing
from simyuj.qstate.state import purity
from simyuj.runtime.binding import BindingContext

## 1. Report sinks

The quantum components store their own reports too, but sinks make the port-delivered reports visible.

In [ ]:
@dataclass(slots=True)
class ReportSink(Component):
    component_id: str
    input_port: Port = field(init=False)
    received: list[tuple[int, object]] = field(default_factory=list)

    def __post_init__(self) -> None:
        self.input_port = Port('in', self, self.component_id, PortKind.CLASSICAL, PortDirection.INGRESS)

    def handle_event(self, event, timeline) -> None:
        delivery = event.payload_ref
        if not isinstance(delivery, PortDelivery):
            raise TypeError('expected PortDelivery')
        self.received.append((timeline.current_time, delivery.payload))


def print_reports(title: str, rows: list[tuple[int, object]]) -> None:
    print(title)
    for time, report in rows:
        outcome = getattr(report, 'outcome', None)
        status = getattr(report, 'status', None)
        success = getattr(report, 'success', None)
        print(' t=', time, '|', type(report).__name__, '| success=', success, '| status=', status, '| outcome=', outcome)

## 2. Build the devices

Both channels have distance and a little noise. The detector is efficient but not perfect. The memory emits notices for absorb and measurement.

In [ ]:
timeline = Timeline(master_seed=91)

source = EntangledPairSource(
    device_id='eps.lab',
    frequency_hz=1e12,
    emission_probability=1.0,
    duration_s=1e-12,
)


In [ ]:
left_channel = QuantumChannel(
    channel_id='fiber.left.to.memory',
    length_m=1_200,
    attenuation_db_per_km=0.05,
    timing_jitter_stddev_ticks=1.0,
    noise_models=(depolarizing(0.03),),
)
right_channel = QuantumChannel(
    channel_id='fiber.right.to.detector',
    length_m=1_600,
    attenuation_db_per_km=0.05,
    timing_jitter_stddev_ticks=1.0,
    noise_models=(depolarizing(0.03),),
)


In [ ]:
memory = QuantumMemory(
    memory_id='nodeA.mem0',
    num_positions=1,
    storage_lifetime_ticks=50_000_000,
    measure_delay_ticks=2,
)
detector = DetectorArray(
    device_id='nodeB.detector',
    detectors=(
        SinglePhotonDetector('d0', SinglePhotonDetectorParams(efficiency=0.90, dark_count_rate_hz=1e9, jitter_stddev_ticks=1.0)),
        SinglePhotonDetector('d1', SinglePhotonDetectorParams(efficiency=0.90, dark_count_rate_hz=1e9, jitter_stddev_ticks=1.0)),
    ),
    measurement='z',
    readout={'z': {'0': 'd0', '1': 'd1'}},
    detection_window_ticks=5,
    output_latency_ticks=1,
)


In [ ]:
memory_reports = ReportSink('memory.report.sink')
detector_reports = ReportSink('detector.report.sink')
source_reports = ReportSink('source.report.sink')


In [ ]:
for component in (left_channel, right_channel, memory, detector):
    component.bind(BindingContext(timeline=timeline, logger=timeline.logger))

connect_ports(source.left_output_port, left_channel.input_port, target_action=ACTION_TRANSMIT_QUANTUM)
connect_ports(left_channel.output_port, memory.input_port, target_action=MEMORY_ABSORB)
connect_ports(source.right_output_port, right_channel.input_port, target_action=ACTION_TRANSMIT_QUANTUM)
connect_ports(right_channel.output_port, detector.input_port, target_action=ACTION_DETECT_SIGNAL)
connect_ports(memory.notice_port, memory_reports.input_port, target_action='memory_notice')
connect_ports(detector.output_port, detector_reports.input_port, target_action='detector_report')
connect_ports(source.report_port, source_reports.input_port, target_action='source_report')

print('left channel delay:', left_channel.resolved_delay_ticks, 'survival:', round(left_channel.survival_probability, 3))
print('right channel delay:', right_channel.resolved_delay_ticks, 'survival:', round(right_channel.survival_probability, 3))
print('memory positions:', len(memory.positions))

## 3. Run the pair emission through the network

One source event causes several downstream events: pair member deliveries, channel forwarding, memory absorb, detector report, and source report.

In [ ]:
source.schedule_start(timeline)
timeline.run_until(10_000_020)

print('timeline stats:', timeline.stats)
print('source reports:', len(source.reports))
print('left channel counts:', {'received': left_channel.received_count, 'delivered': left_channel.delivered_count, 'lost': left_channel.lost_count})
print('right channel counts:', {'received': right_channel.received_count, 'delivered': right_channel.delivered_count, 'lost': right_channel.lost_count})
print('memory report count:', len(memory.reports))
print('detector report count:', len(detector.reports))
print('qstate records:', timeline.qstate.size())

In [ ]:
print_reports('source report sink:', source_reports.received)
print_reports('\nmemory report sink:', memory_reports.received)
print_reports('\ndetector report sink:', detector_reports.received)

## 4. Inspect the memory side

If the left member survived, the memory position owns a stable memory subsystem now.

In [ ]:
for position in memory.positions:
    print('position', position.position, 'status=', position.status.value, 'ready_at=', position.ready_at, 'token=', position.occupancy_token)

if memory.positions[0].status.value == 'occupied':
    subsystem = memory.positions[0].memory_subsystem
    state_ref = timeline.qstate.state_of(subsystem)
    record = timeline.qstate.record(state_ref)
    print('memory subsystem:', subsystem)
    print('state ref:', state_ref)
    print('record rep:', record.rep)
    print('record subsystems:', tuple(str(s) for s in record.layout.subsystems))
    print('payload purity:', round(purity(record.payload), 3))
else:
    print('left member did not end in memory this run')

## 5. Measure the memory after the detector has reported

The memory readout is another event. It targets a physical position, not a raw qstate ref.

In [ ]:
if memory.positions[0].status.value == 'occupied':
    request = MemoryMeasureRequest(
        request_id='measure-memory-after-herald',
        memory_id=memory.memory_id,
        positions=(0,),
        measurement='z',
        destructive=True,
        meta=(('reason', 'after detector report'),),
    )
    timeline.schedule(Event(time=timeline.current_time + 1, target_ref=memory, action=MEMORY_MEASURE, payload_ref=request))
    timeline.run_until_empty()

print('final memory reports:')
for report in memory.reports:
    print(type(report).__name__, 'time=', report.time, 'success=', report.success, 'status=', getattr(report, 'status', None))

print('final positions:')
for position in memory.positions:
    print(position.position, position.status.value, 'ready_at=', position.ready_at)
print('final qstate records:', timeline.qstate.size())

## Keep this model in your head

The run is small, but the same pattern scales: ports wire components, channels change timing and survival, devices mutate qstate through explicit events, and reports are the classical trail you inspect afterward.